# Transformers: scaled dot-product attention

Calculate attention on two tokens. This minimal implementation handles a causal mask and checks that allowed weights sum to one. No external packages are needed.

In [1]:
import math

def softmax(scores):
    peak = max(scores)
    values = [math.exp(score-peak) if score != -math.inf else 0.0 for score in scores]
    total = sum(values)
    return [value/total for value in values]

def attention(queries, keys, values, causal=False):
    dimension = len(keys[0])
    weights, outputs = [], []
    for i, query in enumerate(queries):
        scores = [sum(a*b for a,b in zip(query,key))/math.sqrt(dimension)
                  if not causal or j <= i else -math.inf
                  for j,key in enumerate(keys)]
        row = softmax(scores)
        weights.append(row)
        outputs.append([sum(p*v[k] for p,v in zip(row,values))
                        for k in range(len(values[0]))])
    return weights, outputs

## Compare bidirectional and causal masks

With identity query, key, and value vectors, the outputs equal attention weights. A causal mask removes future tokens before the softmax.

In [2]:
tokens = [[1.0,0.0],[0.0,1.0]]
for causal in (False,True):
    weights, outputs = attention(tokens,tokens,tokens,causal=causal)
    print('causal =', causal)
    for row, out in zip(weights,outputs):
        print(' weights:', [round(v,4) for v in row],
              'output:', [round(v,4) for v in out])
        assert abs(sum(row)-1) < 1e-12

causal = False
 weights: [0.6698, 0.3302] output: [0.6698, 0.3302]
 weights: [0.3302, 0.6698] output: [0.3302, 0.6698]
causal = True
 weights: [1.0, 0.0] output: [1.0, 0.0]
 weights: [0.3302, 0.6698] output: [0.3302, 0.6698]


## What the scale changes

Without division by $\sqrt{d_k}$, larger dot products often make softmax sharper. This tiny example demonstrates only the numerical effect, not a model quality result.

In [3]:
for scale in (1.0, math.sqrt(2)):
    weights = softmax([1/scale, 0])
    print(f'scale={scale:.4f}:', [round(v,4) for v in weights])

scale=1.0000: [0.7311, 0.2689]
scale=1.4142: [0.6698, 0.3302]


### Try it

Change the keys, compare masked and unmasked token 1, and add a third token. Verify that a future token has exactly zero weight in a causal row. A production implementation also needs padding masks, multiple learned heads, and numerical care.